# Workflow Reservoir Simulator Development

Notebook ini menerjemahkan `docs/workflow.md` ke alur kerja yang bisa dilihat step by step. Struktur cell sengaja mengikuti 6 tahap utama workflow simulator reservoir:

1. Preparation
2. Connection List
3. Residual Calculation
4. Jacobian of Residual Function
5. Update Iteration
6. Check Residual and Numerical Constraints

Data kasus diambil dari project, yaitu `manual_engine/run_manual.py`. Semua data penting dijaga dalam bentuk `DataFrame` supaya hubungan antar cell, properti, flux, accumulation, residual, dan update Newton tetap nyambung dalam satu alur kerja yang sama.

In [ ]:
from __future__ import annotations

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

import manual_engine.run_manual as rm

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda value: f'{value:,.6f}')
rm.VERBOSE_NEWTON = False

PVT_TABLES = {
    'bo': rm.PVT_BO,
    'bw': rm.PVT_BW,
    'bg': rm.PVT_BG,
    'mu_o': rm.PVT_MU_O,
    'mu_w': rm.PVT_MU_W,
    'mu_g': rm.PVT_MU_G,
}

ROCK_TABLES = {
    'kro': rm.ROCK_KRO,
    'krw': rm.ROCK_KRW,
    'krg': rm.ROCK_KRG,
    'pcow': rm.ROCK_PCOW,
    'pcgw': rm.ROCK_PCGW,
}

grid = rm.build_grid()

def make_lookup_df(table_name: str, rows: list[tuple[float, float]], x_name: str, y_name: str) -> pd.DataFrame:
    return pd.DataFrame(rows, columns=[x_name, y_name]).assign(table=table_name)

def state_to_df(state: dict, label: str) -> pd.DataFrame:
    df = pd.DataFrame({
        'cell': np.arange(1, len(state['p']) + 1),
        'pressure_psia': state['p'],
        'sw': state['sw'],
        'sg': state['sg'],
    })
    df['so'] = 1.0 - df['sw'] - df['sg']
    df['state'] = label
    return df[['state', 'cell', 'pressure_psia', 'sw', 'sg', 'so']]

def evaluate_state(grid: dict, state: dict) -> pd.DataFrame:
    rows = []
    for cell in grid['cells']:
        idx = cell['id']
        props = rm.cell_props(state['p'][idx], state['sw'][idx], state['sg'][idx])
        rows.append({
            'cell': idx + 1,
            'pressure_psia': state['p'][idx],
            'sw': state['sw'][idx],
            'sg': state['sg'][idx],
            'so': props['so'],
            'depth_ft': cell['depth'],
            'bulk_vol_ft3': cell['bulk_vol'],
            'poro': cell['poro'],
            'pore_volume_ft3': rm.pore_volume(cell, state['p'][idx]),
            'bo': props['bo'],
            'bw': props['bw'],
            'bg': props['bg'],
            'mu_o': props['mu_o'],
            'mu_w': props['mu_w'],
            'mu_g': props['mu_g'],
            'kro': props['kro'],
            'krw': props['krw'],
            'krg': props['krg'],
            'pcow': props['pcow'],
            'pcgw': props['pcgw'],
            'rho_o': props['rho_o'],
            'rho_w': props['rho_w'],
            'rho_g': props['rho_g'],
            'lam_o': props['lam_o'],
            'lam_w': props['lam_w'],
            'lam_g': props['lam_g'],
        })
    return pd.DataFrame(rows)

def jacobian_labels(n_cells: int) -> tuple[list[str], list[str]]:
    row_labels = [f'R_o[{i}]' for i in range(1, n_cells + 1)]
    row_labels += [f'R_w[{i}]' for i in range(1, n_cells + 1)]
    row_labels += [f'R_g[{i}]' for i in range(1, n_cells + 1)]
    col_labels = [f'p[{i}]' for i in range(1, n_cells + 1)]
    col_labels += [f'Sw[{i}]' for i in range(1, n_cells + 1)]
    col_labels += [f'Sg[{i}]' for i in range(1, n_cells + 1)]
    return row_labels, col_labels

def show(fig):
    fig.tight_layout()
    display(fig)
    plt.close(fig)


## Step 1 - Preparation

Sesuai `3. Tahap 1 - Preparation` di dokumen, tahap ini menyiapkan semua input yang dibutuhkan sebelum aliran dihitung. Intinya simulator harus tahu:

- bentuk dan ukuran grid,
- properti batuan per cell,
- tabel PVT dan rock-fluid,
- kondisi awal,
- serta parameter solver untuk loop waktu dan Newton iteration.

Di bawah ini, semua input itu diringkas ke beberapa `DataFrame` lalu divisualisasikan supaya kelihatan hubungan data lookup dengan model grid yang akan dihitung.

In [ ]:
df_reference = pd.DataFrame([
    {
        'reference_depth_ft': rm.REFERENCE_DEPTH,
        'reference_pressure_psia': rm.P_REF,
        'rho_oil_ref_lbm_ft3': rm.RHO_OIL_REF,
        'rho_water_ref_lbm_ft3': rm.RHO_WATER_REF,
        'rho_gas_ref_lbm_ft3': rm.RHO_GAS_REF,
        'rock_compressibility_psi_inv': rm.ROCK_COMPRESS,
    }
])

df_grid_spec = pd.DataFrame([
    {
        'nx': rm.NX,
        'ny': rm.NY,
        'nz': rm.NZ,
        'dx_ft': rm.DX,
        'dy_ft': rm.DY,
        'dz_ft': rm.DZ,
        'ngrid': rm.NX * rm.NY * rm.NZ,
        'porosity': rm.POROSITY,
        'perm_x_md': rm.PERM_X,
        'perm_y_md': rm.PERM_Y,
        'perm_z_md': rm.PERM_Z,
    }
])

df_solver = pd.DataFrame([
    {
        'dt_initial_day': rm.DT_INITIAL,
        'dt_min_day': rm.DT_MIN,
        'max_time_day': rm.MAX_TIME,
        'growth_factor': rm.GROWTH_FACTOR,
        'shrink_factor': rm.SHRINK_FACTOR,
        'max_retries': rm.MAX_RETRIES,
        'max_newton_iter': rm.MAX_NEWTON_ITER,
        'residual_tol': rm.RESID_TOL,
        'parameter_tol': rm.PARAM_TOL,
    }
])

df_pvt = pd.concat(
    [make_lookup_df(name, rows, 'pressure_psia', 'value') for name, rows in PVT_TABLES.items()],
    ignore_index=True,
)[['table', 'pressure_psia', 'value']]

df_rock = pd.concat(
    [make_lookup_df(name, rows, 'saturation', 'value') for name, rows in ROCK_TABLES.items()],
    ignore_index=True,
)[['table', 'saturation', 'value']]

df_cells = pd.DataFrame(grid['cells']).rename(columns={
    'id': 'cell_id',
    'i': 'i_index',
    'j': 'j_index',
    'k': 'k_index',
    'depth': 'depth_ft',
    'bulk_vol': 'bulk_vol_ft3',
    'kx': 'kx_md',
    'ky': 'ky_md',
    'kz': 'kz_md',
}).assign(cell=lambda d: d['cell_id'] + 1)
df_cells = df_cells[['cell', 'cell_id', 'i_index', 'j_index', 'k_index', 'depth_ft', 'bulk_vol_ft3', 'poro', 'kx_md', 'ky_md', 'kz_md']]

display(df_reference)
display(df_grid_spec)
display(df_solver)
display(df_cells.head())
display(df_pvt.head(12))
display(df_rock.head(12))

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, table_name in zip(axes.flatten(), PVT_TABLES):
    table_df = df_pvt[df_pvt['table'] == table_name]
    ax.plot(table_df['pressure_psia'], table_df['value'], marker='o')
    ax.set_title(table_name.upper())
    ax.set_xlabel('Pressure (psia)')
    ax.set_ylabel('Value')
    ax.grid(True, alpha=0.3)
show(fig)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, group in zip(axes, [('kro', 'krw', 'pcow'), ('krg', 'pcgw')]):
    for table_name in group:
        table_df = df_rock[df_rock['table'] == table_name]
        ax.plot(table_df['saturation'], table_df['value'], marker='o', label=table_name)
    ax.set_xlabel('Saturation')
    ax.set_ylabel('Value')
    ax.grid(True, alpha=0.3)
    ax.legend()
axes[0].set_title('Oil-Water Tables')
axes[1].set_title('Gas-Water Tables')
show(fig)


## Step 2 - Connection List

Sesuai `4. Tahap 2 - Connection List`, simulator harus tahu cell mana yang bertetangga dan seberapa kuat hubungan alirannya. Maka setiap koneksi menyimpan minimal:

- `from_cell` dan `to_cell`,
- arah koneksi,
- luas interface `A`,
- jarak pusat-ke-pusat `L`,
- transmissibility `T`.

Visualisasi di bawah menunjukkan geometri sederhana grid dan garis koneksi antar cell, agar connection list tidak terasa abstrak.

In [ ]:
perm_map = {'x': rm.PERM_X, 'y': rm.PERM_Y, 'z': rm.PERM_Z}

df_connections = pd.DataFrame(grid['connections']).copy()
df_connections['connection'] = np.arange(1, len(df_connections) + 1)
df_connections['from_cell'] = df_connections['from'] + 1
df_connections['to_cell'] = df_connections['to'] + 1
df_connections['harmonic_perm_md'] = df_connections['dir'].map(perm_map)
df_connections['transmissibility_check'] = (
    rm.TRANSMISSIBILITY_UNIT_FACTOR
    * df_connections['harmonic_perm_md']
    * df_connections['area']
    / df_connections['dist']
)
df_connections = df_connections[[
    'connection', 'from_cell', 'to_cell', 'dir', 'area', 'dist', 'harmonic_perm_md', 'T', 'transmissibility_check'
]]

df_connection_count = pd.concat([
    df_connections[['from_cell']].rename(columns={'from_cell': 'cell'}),
    df_connections[['to_cell']].rename(columns={'to_cell': 'cell'})
], ignore_index=True).groupby('cell').size().reset_index(name='n_connections')

display(df_connections.head(12))
display(df_connection_count.sort_values('cell').reset_index(drop=True))

df_grid_plot = df_cells.assign(x=df_cells['i_index'], y=-df_cells['j_index'])
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for _, conn in df_connections.iterrows():
    p1 = df_grid_plot.loc[df_grid_plot['cell'] == conn['from_cell'], ['x', 'y']].iloc[0]
    p2 = df_grid_plot.loc[df_grid_plot['cell'] == conn['to_cell'], ['x', 'y']].iloc[0]
    axes[0].plot([p1['x'], p2['x']], [p1['y'], p2['y']], color='tab:blue', alpha=0.6)
axes[0].scatter(df_grid_plot['x'], df_grid_plot['y'], s=180, color='white', edgecolor='black')
for _, row in df_grid_plot.iterrows():
    axes[0].text(row['x'], row['y'], int(row['cell']), ha='center', va='center')
axes[0].set_title('Grid Cells and Connections')
axes[0].set_xlabel('i-index')
axes[0].set_ylabel('j-index (flipped)')
axes[0].grid(True, alpha=0.3)
axes[0].set_aspect('equal', adjustable='box')

axes[1].bar(df_connection_count['cell'], df_connection_count['n_connections'], color='tab:orange')
axes[1].set_title('Number of Connections per Cell')
axes[1].set_xlabel('Cell')
axes[1].set_ylabel('n_connections')
axes[1].grid(True, axis='y', alpha=0.3)
show(fig)


## Step 3 - Residual Calculation

Sesuai `5. Tahap 3 - Residual Calculation`, residual dibentuk dari kombinasi:

- state lama `n`,
- state tebakan Newton `k`,
- properti PVT dan rock-fluid pada state sekarang,
- flux antar koneksi,
- accumulation per cell.

Di tahap ini kita sengaja menyimpan state, properti, flux, accumulation, dan residual dalam `DataFrame` terpisah, tetapi semua tetap terhubung lewat indeks `cell` atau `connection`.

In [ ]:
dt = rm.DT_INITIAL
state_n = rm.init_state(grid['cells'])
state_k = {
    'p': [max(14.7, pressure - 2.0) for pressure in state_n['p']],
    'sw': list(state_n['sw']),
    'sg': list(state_n['sg']),
}

df_state_n = state_to_df(state_n, 'n')
df_state_k = state_to_df(state_k, 'k')
df_props_k = evaluate_state(grid, state_k)

net_flux = rm.compute_net_flux(grid, state_k)
accumulation = rm.compute_accumulation(grid, state_n, state_k, dt)
residual = rm.compute_residual(net_flux, accumulation)
residual_norm = rm.residual_norm(residual, net_flux, accumulation)

df_conn_flux = pd.concat([
    df_connections[['connection', 'from_cell', 'to_cell', 'dir', 'T']].reset_index(drop=True),
    pd.DataFrame(net_flux['conn']).rename(columns={
        'oil': 'flux_oil',
        'water': 'flux_water',
        'gas': 'flux_gas',
        'phi_o': 'potential_oil',
        'phi_w': 'potential_water',
        'phi_g': 'potential_gas',
    })
], axis=1)

df_net_flux = pd.DataFrame({
    'cell': np.arange(1, len(net_flux['oil']) + 1),
    'net_flux_oil': net_flux['oil'],
    'net_flux_water': net_flux['water'],
    'net_flux_gas': net_flux['gas'],
})

df_accumulation = pd.DataFrame({
    'cell': np.arange(1, len(accumulation['oil']) + 1),
    'acc_oil': accumulation['oil'],
    'acc_water': accumulation['water'],
    'acc_gas': accumulation['gas'],
    'acc_total': accumulation['total'],
})

df_residual = pd.DataFrame({
    'cell': np.arange(1, len(residual['oil']) + 1),
    'residual_oil': residual['oil'],
    'residual_water': residual['water'],
    'residual_gas': residual['gas'],
    'residual_total': residual['total'],
})

df_residual_summary = pd.DataFrame([
    {
        'residual_norm': residual_norm,
        'max_abs_residual_oil': np.max(np.abs(df_residual['residual_oil'])),
        'max_abs_residual_water': np.max(np.abs(df_residual['residual_water'])),
        'max_abs_residual_gas': np.max(np.abs(df_residual['residual_gas'])),
    }
])

display(df_state_n.head())
display(df_state_k.head())
display(df_props_k.head())
display(df_conn_flux.head(12))
display(df_net_flux.head())
display(df_accumulation.head())
display(df_residual.head())
display(df_residual_summary)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(df_props_k['cell'], df_props_k['pressure_psia'], marker='o')
axes[0, 0].set_title('Pressure per Cell at State k')
axes[0, 0].set_xlabel('Cell')
axes[0, 0].set_ylabel('Pressure (psia)')
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(df_props_k['cell'], df_props_k['sw'], marker='o', label='Sw')
axes[0, 1].plot(df_props_k['cell'], df_props_k['so'], marker='o', label='So')
axes[0, 1].plot(df_props_k['cell'], df_props_k['sg'], marker='o', label='Sg')
axes[0, 1].set_title('Saturation per Cell at State k')
axes[0, 1].set_xlabel('Cell')
axes[0, 1].set_ylabel('Saturation')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].bar(df_residual['cell'] - 0.25, df_residual['residual_oil'], width=0.25, label='Oil')
axes[1, 0].bar(df_residual['cell'], df_residual['residual_water'], width=0.25, label='Water')
axes[1, 0].bar(df_residual['cell'] + 0.25, df_residual['residual_gas'], width=0.25, label='Gas')
axes[1, 0].set_title('Residual by Phase and Cell')
axes[1, 0].set_xlabel('Cell')
axes[1, 0].set_ylabel('Residual')
axes[1, 0].legend()
axes[1, 0].grid(True, axis='y', alpha=0.3)

axes[1, 1].bar(df_conn_flux['connection'], df_conn_flux['flux_oil'], alpha=0.7, label='Oil')
axes[1, 1].bar(df_conn_flux['connection'], df_conn_flux['flux_water'], alpha=0.7, label='Water')
axes[1, 1].bar(df_conn_flux['connection'], df_conn_flux['flux_gas'], alpha=0.7, label='Gas')
axes[1, 1].set_title('Connection Fluxes')
axes[1, 1].set_xlabel('Connection')
axes[1, 1].set_ylabel('Flux')
axes[1, 1].legend()
axes[1, 1].grid(True, axis='y', alpha=0.3)
show(fig)


## Step 4 - Jacobian of Residual Function

Sesuai `6. Tahap Jacobian`, residual yang nonlinear perlu diterjemahkan menjadi matriks sensitivitas `J = ∂R/∂m`. Di kasus ini unknown per cell adalah `p`, `Sw`, dan `Sg`, sehingga ukuran Jacobian menjadi `3n x 3n`.

Notebook ini memakai Jacobian finite difference dari implementasi project, lalu menampilkan:

- statistik ukuran matriks,
- blok awal Jacobian,
- heatmap magnitude Jacobian.

In [ ]:
jacobian = rm.assemble_jacobian(grid, state_n, state_k, dt)
n_cells = len(grid['cells'])
row_labels, col_labels = jacobian_labels(n_cells)
jacobian_df = pd.DataFrame(jacobian, index=row_labels, columns=col_labels)
jacobian_abs = np.abs(jacobian_df.to_numpy())

df_jacobian_stats = pd.DataFrame([
    {
        'n_rows': jacobian_df.shape[0],
        'n_cols': jacobian_df.shape[1],
        'nnz': int(np.count_nonzero(jacobian_df.to_numpy())),
        'max_abs_value': float(jacobian_abs.max()),
    }
])

display(df_jacobian_stats)
display(jacobian_df.iloc[:9, :9])

fig, ax = plt.subplots(figsize=(8, 7))
image = ax.imshow(np.log10(jacobian_abs + 1e-12), cmap='viridis', aspect='auto')
ax.set_title('Jacobian Heatmap (log10 |J|)')
ax.set_xlabel('Unknown index')
ax.set_ylabel('Residual index')
fig.colorbar(image, ax=ax, label='log10 |value|')
show(fig)


## Step 5 - Update Iteration

Sesuai `7. Tahap Newton update`, sistem linear `J Δm = -R` diselesaikan untuk memperoleh koreksi unknown. Lalu state diperbarui:

- `p^(k+1) = p^k + Δp`
- `Sw^(k+1) = Sw^k + ΔSw`
- `Sg^(k+1) = Sg^k + ΔSg`
- `So^(k+1) = 1 - Sw^(k+1) - Sg^(k+1)`

Di notebook ini, hasil update diperlihatkan sebagai tabel delta dan plot perbandingan state sebelum-sesudah koreksi.

In [ ]:
rhs = [-value for value in residual['vec']]
delta = rm.gauss_solve(jacobian, rhs)
state_k1 = rm.apply_correction(state_k, delta)

df_delta = pd.DataFrame({
    'cell': np.arange(1, n_cells + 1),
    'delta_p': delta[:n_cells],
    'delta_sw': delta[n_cells:2 * n_cells],
    'delta_sg': delta[2 * n_cells:],
})

df_state_k1 = state_to_df(state_k1, 'k+1')
df_update_compare = df_state_k.merge(
    df_state_k1[['cell', 'pressure_psia', 'sw', 'sg', 'so']],
    on='cell',
    suffixes=('_k', '_k1'),
)

display(df_delta.head())
display(df_update_compare.head())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(df_update_compare['cell'], df_update_compare['pressure_psia_k'], marker='o', label='p^k')
axes[0].plot(df_update_compare['cell'], df_update_compare['pressure_psia_k1'], marker='o', label='p^(k+1)')
axes[0].set_title('Pressure Update')
axes[0].set_xlabel('Cell')
axes[0].set_ylabel('Pressure (psia)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(df_update_compare['cell'], df_update_compare['sw_k'], marker='o', label='Sw^k')
axes[1].plot(df_update_compare['cell'], df_update_compare['sw_k1'], marker='o', label='Sw^(k+1)')
axes[1].plot(df_update_compare['cell'], df_update_compare['sg_k'], marker='o', label='Sg^k')
axes[1].plot(df_update_compare['cell'], df_update_compare['sg_k1'], marker='o', label='Sg^(k+1)')
axes[1].set_title('Saturation Update')
axes[1].set_xlabel('Cell')
axes[1].set_ylabel('Saturation')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
show(fig)


## Step 6 - Check Residual and Numerical Constraints

Sesuai `8. Tahap convergence dan constraints`, setelah state di-update kita cek dua hal:

- residual apakah sudah cukup kecil,
- perubahan parameter apakah sudah cukup kecil.

Kalau dua syarat ini lolos, Newton dianggap konvergen dan simulasi boleh maju ke time step berikutnya. Kalau belum, workflow kembali ke residual, Jacobian, dan update iteration. Cell terakhir juga menjalankan satu `run_timestep` penuh agar alur end-to-end project terlihat utuh.

In [ ]:
net_flux_k1 = rm.compute_net_flux(grid, state_k1)
accumulation_k1 = rm.compute_accumulation(grid, state_n, state_k1, dt)
residual_k1 = rm.compute_residual(net_flux_k1, accumulation_k1)
residual_norm_k1 = rm.residual_norm(residual_k1, net_flux_k1, accumulation_k1)

max_dp_rel = max(
    abs(state_k1['p'][idx] - state_k['p'][idx]) / max(abs(state_k['p'][idx]), 1.0)
    for idx in range(n_cells)
)
max_dsw = max(abs(state_k1['sw'][idx] - state_k['sw'][idx]) for idx in range(n_cells))
max_dsg = max(abs(state_k1['sg'][idx] - state_k['sg'][idx]) for idx in range(n_cells))

df_convergence = pd.DataFrame([
    {
        'residual_norm_before': residual_norm,
        'residual_norm_after': residual_norm_k1,
        'residual_ok': residual_norm_k1 <= rm.RESID_TOL,
        'max_dp_rel': max_dp_rel,
        'max_dsw': max_dsw,
        'max_dsg': max_dsg,
        'parameter_ok': max(max_dp_rel, max_dsw, max_dsg) <= rm.PARAM_TOL,
    }
])

state_step_1, converged, final_norm, used_iter = rm.run_timestep(grid, state_n, dt, dt, 1)
df_step_summary = pd.DataFrame([
    {
        'timestep_day': dt,
        'converged': converged,
        'newton_iterations': used_iter,
        'final_norm': final_norm,
        'next_action': 'advance to next timestep' if converged else 'repeat residual-jacobian-update loop',
    }
])
df_state_step_1 = state_to_df(state_step_1, 'accepted_step')

display(df_convergence)
display(df_step_summary)
display(df_state_step_1.head())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].bar(['before', 'after'], [residual_norm, residual_norm_k1], color=['tab:red', 'tab:green'])
axes[0].axhline(rm.RESID_TOL, color='black', linestyle='--', label='tolerance')
axes[0].set_title('Residual Norm Check')
axes[0].set_ylabel('Norm')
axes[0].legend()

axes[1].bar(['dp_rel', 'dSw', 'dSg'], [max_dp_rel, max_dsw, max_dsg], color='tab:purple')
axes[1].axhline(rm.PARAM_TOL, color='black', linestyle='--', label='tolerance')
axes[1].set_title('Parameter Change Check')
axes[1].legend()

axes[2].plot(df_state_step_1['cell'], df_state_step_1['pressure_psia'], marker='o', label='Pressure')
axes[2].plot(df_state_step_1['cell'], df_state_step_1['sw'], marker='o', label='Sw')
axes[2].plot(df_state_step_1['cell'], df_state_step_1['so'], marker='o', label='So')
axes[2].plot(df_state_step_1['cell'], df_state_step_1['sg'], marker='o', label='Sg')
axes[2].set_title('Accepted State After One Timestep')
axes[2].set_xlabel('Cell')
axes[2].legend()
axes[2].grid(True, alpha=0.3)
show(fig)
